In [1]:
import torch
from torch.utils.data import Dataset
import pickle

AA_TO_INDEX = {
    'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4,
    'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9,
    'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14,
    'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19,
    '-': 20, 'X': 20  # padding 또는 unknown
}

class MSADataset(Dataset):
    def __init__(self, df, msa_dict_path, max_depth=80, win_size=61):
        with open(msa_dict_path, "rb") as f:
            self.msa_dict = pickle.load(f)
        self.df = df
        self.max_depth = max_depth
        self.win_size = win_size
        self.half_win = win_size // 2  # 중심에서 양쪽 길이

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        uid = row["UniProtID"]
        mut_pos = int(row["MutPos"]) - 1  # 1-based → 0-based
        label = int(row["Label"])
        mut = row["Mut"].upper()

        msa_seqs = [seq for _, seq in self.msa_dict[uid][:self.max_depth]]
        query_seq = msa_seqs[0]  # 보통 첫 줄이 ref seq

        # 변이 반영된 mut_seq 생성
        mut_seq = list(query_seq)
        if 0 <= mut_pos < len(mut_seq):
            mut_seq[mut_pos] = mut

        # 사용될 시퀀스: 변이 시퀀스 + ref 시퀀스 + MSA
        seqs_to_use = [mut_seq, list(query_seq)]
        if len(msa_seqs) > 1:
            seqs_to_use += [list(seq) for seq in msa_seqs[1:self.max_depth - 2]]

        centered_msa = []
        for seq in seqs_to_use:
            window = []
            for i in range(self.win_size):
                seq_idx = mut_pos - self.half_win + i
                if 0 <= seq_idx < len(seq):
                    aa = seq[seq_idx]
                else:
                    aa = '-'
                window.append(AA_TO_INDEX.get(aa, 20))
            centered_msa.append(window)

        # depth padding
        while len(centered_msa) < self.max_depth:
            centered_msa.append([20] * self.win_size)  # 20은 패딩 인덱스

        msa_tensor = torch.tensor(centered_msa[:self.max_depth]).long()  # [D, L]
        msa_tensor = msa_tensor.transpose(0, 1)  # [L, D]

        return {
            "msa": msa_tensor,  # [L, D]
            "label": torch.tensor(label).long()
        }


In [2]:
import torch
import torch.nn as nn
from mamba_ssm import Mamba
from mamba_ssm import Mamba2


# --- Input Embedding ---
class MSAInputEmbedding(nn.Module):
    def __init__(self, vocab_size=21, dim=256):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=dim)

    def forward(self, x):  # x: (B, L, D)
        return self.embedding(x)  # → (B, L, D, C)

# --- MambaRMSNorm ---
class MambaRMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x.norm(dim=-1, keepdim=True) / (x.shape[-1] ** 0.5)
        return self.weight * x / (norm + self.eps)

# --- Cross-Axial Mamba Block ---
class CrossAxialMambaMSA(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm_L = MambaRMSNorm(dim)
        self.norm_D = MambaRMSNorm(dim)

        self.mamba_L = Mamba(d_model=dim, expand=1)
        self.conv_D = nn.Conv1d(in_channels=dim, out_channels=dim, kernel_size=5, padding=2)

    def forward(self, x):  # x: (B, L, D, C)
        B, L, D, C = x.shape
        center_L = L // 2  # = 30

        # --- D-axis: only at center L position ---
        x_d_center = self.norm_D(x[:, center_L])  # (B, D, C)
        x_d = x_d_center.transpose(1, 2)          # (B, C, D)
        d_out = self.conv_D(x_d).transpose(1, 2).unsqueeze(1)  # (B, 1, D, C)

        # --- L-axis: full Mamba ---
        x_l = self.norm_L(x).permute(0, 2, 1, 3).contiguous().view(B * D, L, C)
        l_out = self.mamba_L(x_l).view(B, D, L, C).permute(0, 2, 1, 3)  # (B, L, D, C)

        # --- Residual ---
        # d_out is only for center, rest is zero
        d_full = torch.zeros_like(x)
        d_full[:, center_L:center_L+1] = d_out

        return x + d_full + l_out


# # --- Encoder ---
class MSAEncoder(nn.Module):
    def __init__(self, num_layers=8, dim=256):
        super().__init__()
        self.embeddings = MSAInputEmbedding(dim=dim)
        self.blocks = nn.ModuleList([
            CrossAxialMambaMSA(dim) for _ in range(num_layers)
        ])
        self.norm_f = MambaRMSNorm(dim)

    def forward(self, x):  # x: (B, L, D)
        x = self.embeddings(x)  # → (B, L, D, C)
        for block in self.blocks:
            x = block(x)
        return self.norm_f(x)  # (B, L, D, C)

# --- Classifier Head ---
class MSAClassifier(nn.Module):
    def __init__(self, num_layers=4, dim=128, num_classes=2):
        super().__init__()
        self.encoder = MSAEncoder(num_layers=num_layers, dim=dim)

        self.classifier = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, num_classes)
        )

    def forward(self, x):  # x: (B, L, D)
        x = self.encoder(x)                  # (B, L, D, C)
        center_L = x.shape[1] // 2           # 30
        x = x[:, center_L]                   # (B, D, C)
        x = x.mean(dim=1)                    # mean over D → (B, C)
        out = self.classifier(x)             # (B, num_classes)
        return out


/home/kunny/miniconda3/envs/mamba_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model = MSAClassifier(num_layers=4, dim=128, num_classes=2)

In [4]:
# from torchinfo import summary
# import torch

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model = MSAClassifier(num_layers=4, dim=128, num_classes=2).to(device)

# dummy_input = torch.randint(low=0, high=21, size=(1, 61, 80)).long().to(device)

# summary(
#     model,
#     input_data=(dummy_input,),
#     col_names=["input_size", "output_size", "num_params"],
#     row_settings=["var_names"]
# )


In [5]:
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.model_selection import KFold

df = pd.read_csv("/mnt/c/Users/Kunny/Research/Dataset/Missense_Variant_dataset/rhapsody2_sav_db_exactmatch_only.tsv", sep="\t", header=None)
df.columns = ["UniProtID", "StructureFile", "MutPos", "WT", "Mut", "Label"]

# 10-fold 
kf = KFold(n_splits=10, shuffle=True, random_state=42)
splits = list(kf.split(df))
train_idx, val_idx = splits[0]

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

# oversampling: label == 1
pos_df = train_df[train_df["Label"] == 1]
neg_df = train_df[train_df["Label"] == 0]

repeat_factor = max(1, len(neg_df) // max(len(pos_df), 1))
oversampled_train_df = pd.concat([neg_df, pd.concat([pos_df] * repeat_factor)], ignore_index=True)
oversampled_train_df = oversampled_train_df.sample(frac=1, random_state=42).reset_index(drop=True)  # 셔플

# Dataset
train_dataset = MSADataset(oversampled_train_df, "/mnt/e/CAGI_data/msa_dict_valid.pkl")
val_dataset   = MSADataset(val_df, "/mnt/e/CAGI_data/msa_dict_valid.pkl")

# 5. Dataloader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, persistent_workers=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True)

In [ ]:
print("=== 오버샘플링 전 ===")
print(f"  원본 train_df: {len(train_df)}")
print(f"    - Label 0 개수: {len(neg_df)}")
print(f"    - Label 1 개수: {len(pos_df)}")

print("\n=== 오버샘플링 후 ===")
print(f"  oversampled_train_df: {len(oversampled_train_df)}")
print(f"    - Label 0 개수: {(oversampled_train_df['Label'] == 0).sum()}")
print(f"    - Label 1 개수: {(oversampled_train_df['Label'] == 1).sum()}")

print(f"  원본 val_df: {len(val_df)}")


=== 오버샘플링 전 ===
  원본 train_df: 89869
    - Label 0 개수: 58355
    - Label 1 개수: 31514

=== 오버샘플링 후 ===
  oversampled_train_df: 89869
    - Label 0 개수: 58355
    - Label 1 개수: 31514
  원본 val_df: 9986


In [7]:
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MSAClassifier(num_layers=8, dim=128).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

num_epochs = 100
best_pr_auc = 0.0
save_path = "/mnt/e/CAGI_data/best_model_250730.pth"

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        x = batch["msa"].to(device)         # [B, L, D]
        y = batch["label"].to(device)       # [B]

        optimizer.zero_grad()
        logits = model(x)                   # [B, 2]
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)

    scheduler.step()
    avg_train_loss = train_loss / len(train_loader.dataset)

    # --- Validation ---
    model.eval()
    val_loss = 0
    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            x = batch["msa"].to(device)
            y = batch["label"].to(device)

            logits = model(x)
            loss = criterion(logits, y)

            probs = torch.softmax(logits, dim=1)[:, 1]  # P(class=1)

            val_loss += loss.item() * x.size(0)
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader.dataset)
    pr_auc = average_precision_score(all_labels, all_probs)

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val PR-AUC: {pr_auc:.4f}")

    # --- Save best model ---
    if pr_auc > best_pr_auc:
        best_pr_auc = pr_auc
        torch.save(model.state_dict(), save_path)
        print(f">>> Best model saved! PR-AUC: {pr_auc:.4f}")


Epoch 1 [Val]: 100%|██████████| 313/313 [00:16<00:00, 18.54it/s]



Epoch 1/100
Train Loss: 0.5118 | Val Loss: 0.4799 | Val PR-AUC: 0.7137
>>> Best model saved! PR-AUC: 0.7137


Epoch 2 [Val]: 100%|██████████| 313/313 [00:15<00:00, 19.85it/s]



Epoch 2/100
Train Loss: 0.4663 | Val Loss: 0.4605 | Val PR-AUC: 0.7474
>>> Best model saved! PR-AUC: 0.7474


Epoch 3 [Val]: 100%|██████████| 313/313 [00:15<00:00, 20.02it/s]


...


Epoch 33/100
Train Loss: 0.0258 | Val Loss: 0.9001 | Val PR-AUC: 0.8487


Epoch 34 [Val]: 100%|██████████| 313/313 [00:16<00:00, 18.43it/s]



Epoch 34/100
Train Loss: 0.0253 | Val Loss: 0.8811 | Val PR-AUC: 0.8555


Epoch 35 [Train]:  40%|███▉      | 1118/2809 [03:33<05:23,  5.23it/s]


KeyboardInterrupt: 